In [2]:
import fitz  # PyMuPDF
import pandas as pd
import re
from pathlib import Path

In [3]:
# Configuración dinámica de rutas relativas al notebook actual
# Asumiendo que el notebook está en /notebooks y los datos en /data
BASE_DIR = Path.cwd().parent 
RAW_PDF_PATH = BASE_DIR / "data" / "raw" / "UDEF-Zapatero-1.pdf"
PROCESSED_CSV_PATH = BASE_DIR / "data" / "processed" / "chats_limpios.csv"

In [26]:
# Verificamos que el archivo raw exista antes de continuar
if not RAW_PDF_PATH.exists():
    raise FileNotFoundError(f"No se encuentra el documento en: {RAW_PDF_PATH}")

# Regex Nivel Avanzado:
# 1. Buscamos el patrón de fecha y hora.
# 2. re.DOTALL permite que el '.*' del mensaje capture saltos de línea (mensajes largos).
# 3. El Lookahead (?=\n\s*\[\d|\Z) indica: "captura el mensaje hasta que veas un salto de línea seguido de un corchete y un número (el siguiente chat), o hasta el final del texto (\Z)".
# Regex Formato 1: [dd/mm/yy, hh:mm:ss] Emisor: Mensaje
regex_f1 = re.compile(
    r'\[(\d{1,2}/\d{1,2}/\d{2,4})[,\s]*(\d{1,2}:\d{2}:\d{2})\]\s*([^:]+?):\s*(.*?)(?=\n\s*\[\d{1,2}/\d{1,2}/\d{2,4}|\Z)', 
    re.DOTALL
)

# Regex F2: 
# Soporta nombres en minúscula, teléfonos, "Fecha" o "Date", y zonas horarias variables.
regex_f2 = re.compile(
    r'\s*(?:([A-Za-zÁÉÍÓÚÑáéíóúñ0-9\+\-\.\(\) ]+)\s*\n)?' 
    r'(?:Timestamp|Fecha|Date)\s*:\s*(\d{1,2}/\d{1,2}/\d{2,4})\s+(\d{1,2}:\d{2}:\d{2})(?:\s*\(UTC[^)]*\))?\s*\n'
    r'(.*?)'
    r'(?=\n\s*(?:[A-Za-zÁÉÍÓÚÑáéíóúñ0-9\+\-\.\(\) ]+\s*\n)?(?:Timestamp|Fecha|Date)\s*:|\n\s*\[\d{1,2}/|\Z)',
    re.DOTALL
)

In [27]:
def extraer_chats_robusto(pdf_path, pag_inicio, pag_fin):
    doc = fitz.open(pdf_path)
    matches_totales = []
    
    for num_pag in range(pag_inicio, pag_fin + 1):
        texto_pagina = doc[num_pag].get_text()
        
        for match in regex_f1.finditer(texto_pagina):
            fecha, hora, emisor, mensaje = match.groups()
            matches_totales.append({
                "Pagina_PDF": num_pag,
                "Formato": "F1",
                "Fecha": fecha.strip(),
                "Hora": hora.strip(),
                "Emisor": emisor.strip(),
                "Mensaje": " ".join(mensaje.split())
            })
            
        ultimo_emisor = "DESCONOCIDO"
        for match in regex_f2.finditer(texto_pagina):
            emisor, fecha, hora, mensaje = match.groups()
            
            if emisor and emisor.strip():
                ultimo_emisor = emisor.strip()
            
            matches_totales.append({
                "Pagina_PDF": num_pag,
                "Formato": "F2",
                "Fecha": fecha.strip(),
                "Hora": hora.strip(),
                "Emisor": ultimo_emisor,
                "Mensaje": " ".join(mensaje.split())
            })
            
    df = pd.DataFrame(matches_totales)
    
    if not df.empty:
        df = df.sort_values(by=["Pagina_PDF", "Fecha", "Hora"]).reset_index(drop=True)
        
    return df

In [28]:
# Extraemos el bloque cronológico completo
df_chats = extraer_chats_robusto(RAW_PDF_PATH, 30, 154)

print(f"Total de mensajes extraídos: {len(df_chats)}")
print(f"Desglose por formato:\n{df_chats['Formato'].value_counts()}\n")

# Cuando confirmes que el número de mensajes es masivo y correcto, descomenta esta línea:
# df_chats.to_csv(BASE_DIR / "data" / "processed" / "chats_limpios.csv", index=False)
display(df_chats.head())

Total de mensajes extraídos: 574
Desglose por formato:
Formato
F2    338
F1    236
Name: count, dtype: int64



,Pagina_PDF,Formato,Fecha,Hora,Emisor,Mensaje
0,30,F1,23/3/20,23:06:48,Miguel Palomero,Que vais a hacer con la línea aérea?
1,30,F1,23/3/20,23:07:01,Miguel Palomero,"Has visto BA, AE y Iberia?"
2,30,F1,23/3/20,23:07:04,Rodolfo Reyes,Aguantando la paliza
3,30,F1,23/3/20,23:07:16,Miguel Palomero,Me imagino
4,30,F1,23/3/20,23:07:41,Rodolfo Reyes,Necesitamos llegar a las ayudas


In [29]:
df_chats.to_csv(BASE_DIR / "data" / "processed" / "chats_limpios.csv", index=False)

In [17]:
# Probamos el rango completo de la 31 a la 35
df_prueba = extraer_chats_rango(RAW_PDF_PATH, 30, 35)

print(f"Total de mensajes extraídos (Páginas 31-35): {len(df_prueba)}")
display(df_prueba.head(20))

Total de mensajes extraídos (Páginas 31-35): 20


,Pagina,Fecha,Hora,Emisor,Mensaje
0,30,23/3/20,23:06:48,Miguel Palomero,Que vais a hacer con la línea aérea?
1,30,23/3/20,23:07:01,Miguel Palomero,"Has visto BA, AE y Iberia?"
2,30,23/3/20,23:07:04,Rodolfo Reyes,Aguantando la paliza
3,30,23/3/20,23:07:16,Miguel Palomero,Me imagino
4,30,23/3/20,23:07:41,Rodolfo Reyes,Necesitamos llegar a las ayudas
5,30,23/3/20,23:07:47,Miguel Palomero,Así es
6,30,23/3/20,23:07:54,Rodolfo Reyes,Somos pequeños. Será mas fácil
7,30,23/3/20,23:07:57,Miguel Palomero,ERTE para empezar
8,30,23/3/20,23:08:14,Rodolfo Reyes,Ya lo hicimos
9,30,23/3/20,23:08:18,Miguel Palomero,Eso es bueno
